<a href="https://colab.research.google.com/github/Hujjathullah-S-T/Chromatic-9-Graphs/blob/master/Permutating_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import random
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "Chromatic 9 Dataset Graph manual.txt"

OUTPUT_FILE = "Chromatic9_Permutation_Augmented.csv"

# Number of graphs to generate FROM EACH ORIGINAL GRAPH
# Example:
#   1  = original graph only
#   10 = original + 9 permutations
#   20 = original + 19 permutations
#   50 = original + 49 permutations
AUGMENTATIONS_PER_GRAPH = 21

# Chromatic number of this dataset
CHROMATIC_NUMBER = 9

# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)


# ============================================================
# PARSE ONE GRAPH BLOCK
# ============================================================

def parse_graph_block(block):
    """
    Parse a graph block of the form:

    ID1-12.56.1:

    1 - 3,4,8,...
    2 - 6,7,4,...
    ...

    Returns:
        {
            "original_id": ...,
            "n_nodes": ...,
            "header_edges": ...,
            "adjacency": {...}
        }
    """

    lines = block.strip().splitlines()

    if not lines:
        return None

    header = lines[0].strip()

    # Handles:
    # ID1-12.56.1
    # ID2-12.57.1
    # Also handles possible typo such as ID29-.14.72.2
    match = re.match(
        r"ID(\d+)-(\d+)\.(\d+)\.(\d+)",
        header
    )

    if not match:
        # Try a more tolerant format
        match = re.match(
            r"ID(\d+)-.*?(\d+)\.(\d+)\.(\d+)",
            header
        )

    if not match:
        print("WARNING: Could not parse header:")
        print(header)
        return None

    original_id = int(match.group(1))
    n_nodes = int(match.group(2))
    header_edges = int(match.group(3))

    adjacency = {}

    for line in lines[1:]:

        line = line.strip()

        if not line:
            continue

        # Match:
        # 1 - 3,4,8,...
        # 2 – 6,7,4,...
        match_node = re.match(
            r"(\d+)\s*[-–]\s*(.*)",
            line
        )

        if not match_node:
            continue

        node = int(match_node.group(1))

        neighbor_text = match_node.group(2)

        neighbors = []

        for x in neighbor_text.split(","):

            x = x.strip()

            if x.isdigit():
                neighbors.append(int(x))

        adjacency[node] = neighbors

    return {
        "original_id": original_id,
        "n_nodes": n_nodes,
        "header_edges": header_edges,
        "adjacency": adjacency
    }


# ============================================================
# CONVERT ADJACENCY LIST TO UNDIRECTED EDGE SET
# ============================================================

def adjacency_to_edges(adjacency, n_nodes):
    """
    Convert 1-based adjacency lists into an undirected
    0-based edge set.

    Example:

        1 -- 3

    becomes:

        (0, 2)

    Duplicate edges are automatically removed.
    """

    edges = set()

    for u, neighbors in adjacency.items():

        u0 = u - 1

        if u0 < 0 or u0 >= n_nodes:
            continue

        for v in neighbors:

            v0 = v - 1

            if v0 < 0 or v0 >= n_nodes:
                continue

            # Ignore self loops
            if u0 == v0:
                continue

            edge = tuple(sorted((u0, v0)))

            edges.add(edge)

    return edges


# ============================================================
# APPLY NODE PERMUTATION
# ============================================================

def permute_edges(edges, permutation):
    """
    Apply a node permutation.

    permutation[old_node] = new_node

    Example:

        old edge: 0-3

        permutation:
        0 -> 5
        3 -> 2

        new edge:
        2-5
    """

    new_edges = set()

    for u, v in edges:

        new_u = permutation[u]
        new_v = permutation[v]

        if new_u == new_v:
            continue

        new_edges.add(
            tuple(sorted((new_u, new_v)))
        )

    return new_edges


# ============================================================
# EDGE LIST STRING
# ============================================================

def edge_list_to_string(edges):
    """
    Convert:

        {(0,1), (0,3), (1,4)}

    into:

        0-1;0-3;1-4
    """

    sorted_edges = sorted(edges)

    return ";".join(
        f"{u}-{v}"
        for u, v in sorted_edges
    )


# ============================================================
# UPPER TRIANGULAR ADJACENCY
# ============================================================

def upper_triangular_1d(edges, n_nodes):
    """
    Create a 1D upper-triangular adjacency representation.

    Order:

        (0,1), (0,2), ..., (0,n-1),
        (1,2), (1,3), ..., (1,n-1),
        ...

    Number of values:

        n_nodes * (n_nodes - 1) / 2
    """

    edge_set = set(edges)

    values = []

    for i in range(n_nodes):

        for j in range(i + 1, n_nodes):

            if (i, j) in edge_set:
                values.append(1)
            else:
                values.append(0)

    return ",".join(map(str, values))


# ============================================================
# CREATE RANDOM PERMUTATION
# ============================================================

def generate_random_permutation(n_nodes):
    """
    Generate:

        old node -> new node

    Example for 5 nodes:

        [3,0,4,1,2]

    means:

        0 -> 3
        1 -> 0
        2 -> 4
        3 -> 1
        4 -> 2
    """

    permutation = list(range(n_nodes))

    random.shuffle(permutation)

    return permutation


# ============================================================
# PROCESS DATASET
# ============================================================

def main():

    print("=" * 70)
    print("CHROMATIC GRAPH NODE-PERMUTATION DATA AUGMENTATION")
    print("=" * 70)

    # --------------------------------------------------------
    # READ FILE
    # --------------------------------------------------------

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        text = f.read()

    # --------------------------------------------------------
    # SPLIT INTO GRAPH BLOCKS
    # --------------------------------------------------------

    blocks = re.split(
        r"(?=ID\d+-)",
        text
    )

    graphs = []

    for block in blocks:

        if not block.strip():
            continue

        graph = parse_graph_block(block)

        if graph is not None:
            graphs.append(graph)

    print()
    print("Original graphs found :", len(graphs))
    print(
        "Augmentations/graph   :",
        AUGMENTATIONS_PER_GRAPH
    )

    # --------------------------------------------------------
    # OUTPUT STORAGE
    # --------------------------------------------------------

    output_rows = []

    generated_id = 1

    # --------------------------------------------------------
    # PROCESS EACH GRAPH
    # --------------------------------------------------------

    for graph_index, graph in enumerate(graphs):

        n_nodes = graph["n_nodes"]

        original_id = graph["original_id"]

        adjacency = graph["adjacency"]

        # Convert to 0-based undirected edges
        original_edges = adjacency_to_edges(
            adjacency,
            n_nodes
        )

        actual_edges = len(original_edges)

        header_edges = graph["header_edges"]

        # ----------------------------------------------------
        # WARNING IF HEADER DOES NOT MATCH PARSED GRAPH
        # ----------------------------------------------------

        if actual_edges != header_edges:

            print(
                f"WARNING: Graph ID{original_id}: "
                f"header says {header_edges} edges, "
                f"but parsed adjacency gives {actual_edges} "
                f"unique undirected edges."
            )

        # ----------------------------------------------------
        # KEEP TRACK OF PERMUTATIONS
        # ----------------------------------------------------

        seen_graphs = set()

        successful = 0

        attempts = 0

        max_attempts = AUGMENTATIONS_PER_GRAPH * 100

        while (
            successful < AUGMENTATIONS_PER_GRAPH
            and attempts < max_attempts
        ):

            attempts += 1

            # ------------------------------------------------
            # First graph = identity/original
            # ------------------------------------------------

            if successful == 0:

                permutation = list(range(n_nodes))

            else:

                permutation = generate_random_permutation(
                    n_nodes
                )

            # ------------------------------------------------
            # Apply permutation
            # ------------------------------------------------

            permuted_edges = permute_edges(
                original_edges,
                permutation
            )

            # ------------------------------------------------
            # Create unique graph signature
            # ------------------------------------------------

            signature = tuple(
                sorted(permuted_edges)
            )

            # Skip if permutation produced the same
            # labelled graph as one already generated
            if signature in seen_graphs:
                continue

            seen_graphs.add(signature)

            # ------------------------------------------------
            # Calculate values
            # ------------------------------------------------

            n_edges = len(permuted_edges)

            edge_list = edge_list_to_string(
                permuted_edges
            )

            upper_triangular = upper_triangular_1d(
                permuted_edges,
                n_nodes
            )

            # ------------------------------------------------
            # Add row
            # ------------------------------------------------

            output_rows.append({

                "generated_id":
                    generated_id,

                "n_nodes":
                    n_nodes,

                "n_edges":
                    n_edges,

                "chromatic_number":
                    CHROMATIC_NUMBER,

                "edge_list":
                    edge_list,

                "adjacency_upper_triangular":
                    upper_triangular
            })

            generated_id += 1
            successful += 1

        print(
            f"Graph ID{original_id}: "
            f"{successful} versions generated"
        )

    # ========================================================
    # SAVE CSV
    # ========================================================

    df = pd.DataFrame(
        output_rows,
        columns=[
            "generated_id",
            "n_nodes",
            "n_edges",
            "chromatic_number",
            "edge_list",
            "adjacency_upper_triangular"
        ]
    )

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("DATASET AUGMENTATION COMPLETE")
    print("=" * 70)

    print(
        "Original graphs          :",
        len(graphs)
    )

    print(
        "Augmentations per graph  :",
        AUGMENTATIONS_PER_GRAPH
    )

    print(
        "Total generated graphs   :",
        len(df)
    )

    print(
        "Output file              :",
        OUTPUT_FILE
    )

    print()
    print("Columns:")
    print(df.columns.tolist())

    print()
    print("First 5 rows:")
    print(df.head())


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

CHROMATIC GRAPH NODE-PERMUTATION DATA AUGMENTATION

Original graphs found : 40
Augmentations/graph   : 21
Graph ID1: 21 versions generated
Graph ID2: 21 versions generated
Graph ID3: 21 versions generated
Graph ID4: 21 versions generated
Graph ID5: 21 versions generated
Graph ID6: 21 versions generated
Graph ID7: 21 versions generated
Graph ID8: 21 versions generated
Graph ID9: 21 versions generated
Graph ID10: 21 versions generated
Graph ID11: 21 versions generated
Graph ID12: 21 versions generated
Graph ID13: 21 versions generated
Graph ID14: 21 versions generated
Graph ID15: 21 versions generated
Graph ID16: 21 versions generated
Graph ID17: 21 versions generated
Graph ID18: 21 versions generated
Graph ID19: 21 versions generated
Graph ID20: 21 versions generated
Graph ID21: 21 versions generated
Graph ID22: 21 versions generated
Graph ID23: 21 versions generated
Graph ID24: 21 versions generated
Graph ID25: 21 versions generated
Graph ID26: 21 versions generated
Graph ID27: 21 vers